# SciAgent-Bench: GPU experiments (Kaggle)

Runs every GPU-dependent experiment of the benchmark and writes a self-contained results bundle.

**Notebook settings (right-hand panel):** Accelerator **GPU T4 x2**, Internet **On**, Persistence **Files only**
(so an interrupted session can resume; finished steps are skipped).

| step | what it measures | T4 x2 time (full / quick) |
|---|---|---|
| tests | the full test suite, with the torch kernels on the GPU | 3 min |
| verify | reference optima: 20M-draw Monte-Carlo, heat grids to 2048², optimiser robustness | 15 / 5 min |
| gpu_bench | NumPy/SciPy vs torch-CUDA kernel throughput | 5 / 2 min |
| bench_cpu | classical planners, 20 seeds x 2 budgets, statistics | 20 / 5 min |
| surrogate | data curation, GPU training, distillation, int8 | 15 / 3 min |
| bench_llm | open-model LLM planner (with / without GP advisor) vs baselines | 90 / 20 min |
| traces | teacher planning traces for distillation | 90 / 15 min |
| train_student | LoRA distillation of a 0.5B planner on the second T4 | 30 / 10 min |
| inference | latency, throughput, JSON validity: teacher vs base vs distilled (fp16, 4-bit) | 15 / 5 min |
| bench_distill | planning quality of the distilled model on held-out seeds | 60 / 15 min |
| report | technical report + zip bundle | 1 min |

Set `PROFILE = "quick"` for a ~1.5 h run that exercises everything with fewer seeds and a 1.5B teacher.

In [ ]:
PROFILE = "kaggle-t4"          # or "quick"
REPO_URL = "https://github.com/SyedaRubbani/sciagent-bench.git"
RESULTS = "/kaggle/working/results"
import os, glob, subprocess, sys
src = sorted(glob.glob("/kaggle/input/*/sciagent-bench")) or sorted(glob.glob("/kaggle/input/sciagent-bench*"))
if src:   # code uploaded as a Kaggle dataset
    !cp -r {src[0]} /kaggle/working/sciagent-bench
elif not os.path.exists("/kaggle/working/sciagent-bench"):
    !git clone -q {REPO_URL} /kaggle/working/sciagent-bench
%cd /kaggle/working/sciagent-bench
os.environ["SCIAGENT_VLLM_VENV"] = "/tmp/vllm-env"   # vLLM gets its own venv (not saved to output); the kernel torch is untouched

In [ ]:
!nvidia-smi -L
!pip install -q -e ".[dev,surrogate,mcp,plot]" "transformers>=4.44" "peft>=0.12" accelerate bitsandbytes hypothesis

## Run the pipeline
Each cell is one resumable step. Outputs go to `RESULTS/<step>/`; failures are written to `ERROR.txt` and do not stop later independent steps.

### env tests
Environment record and the test suite (torch kernels on CUDA).

In [ ]:
!sciagent experiments --profile $PROFILE --results $RESULTS --steps env tests

### verify
Reference solutions with uncertainty (GPU Monte-Carlo and matrix-free PCG).

In [ ]:
!sciagent experiments --profile $PROFILE --results $RESULTS --steps verify

### gpu_bench
Kernel throughput.

In [ ]:
!sciagent experiments --profile $PROFILE --results $RESULTS --steps gpu_bench

### bench_cpu
Classical planners: 20 seeds, two budgets, bootstrap CIs and paired tests.

In [ ]:
!sciagent experiments --profile $PROFILE --results $RESULTS --steps bench_cpu

### surrogate
Surrogate: curated data, GPU training, distillation, int8.

In [ ]:
!sciagent experiments --profile $PROFILE --results $RESULTS --steps surrogate

### bench_llm
Open-model LLM planner benchmark (vLLM if it installs, transformers otherwise).

In [ ]:
!sciagent experiments --profile $PROFILE --results $RESULTS --steps bench_llm

### traces
Teacher traces on training seeds 1000+ (disjoint from evaluation seeds).

In [ ]:
!sciagent experiments --profile $PROFILE --results $RESULTS --steps traces

### train_student
LoRA distillation of the 0.5B student.

In [ ]:
!sciagent experiments --profile $PROFILE --results $RESULTS --steps train_student

### inference
Serving efficiency: teacher, base, distilled fp16 and 4-bit.

In [ ]:
!sciagent experiments --profile $PROFILE --results $RESULTS --steps inference

### bench_distill
Planning quality of the distilled model on held-out seeds.

In [ ]:
!sciagent experiments --profile $PROFILE --results $RESULTS --steps bench_distill

### report
Technical report and results bundle.

In [ ]:
!sciagent experiments --profile $PROFILE --results $RESULTS --steps report

## Read the report

In [ ]:
from IPython.display import Markdown, display
display(Markdown(open(f"{RESULTS}/report/TECHNICAL_REPORT.md").read()))
print(open(f"{RESULTS}/experiments_status.json").read())

The bundle `results_bundle.zip` (next to the results folder) contains every raw result, trace, cassette and log. Download it and run `sciagent report results/cpu <unzipped>/results --out docs/TECHNICAL_REPORT.md` in the repo to merge GPU and CPU results into one report.